In [96]:
import pandas as pd
import numpy as np

In [97]:
df = pd.read_csv('../data/nhpc_data.csv')

In [98]:
df.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
df.head()

,Date,Close,High,Low,Open,Volume
0,2023-03-06,37.832508,38.542755,37.074912,37.832508,11198659
1,2023-03-08,39.442402,39.584449,37.785158,37.785158,16875862
2,2023-03-09,38.637447,39.679145,38.400698,39.679145,5386529
3,2023-03-10,38.874199,39.158297,37.785155,38.353352,5220866
4,2023-03-13,38.116600,38.968898,37.785153,38.826847,6647383


In [99]:
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)
df.reset_index(drop = True, inplace=True)

In [100]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    700 non-null    datetime64[ns]
 1   Close   700 non-null    float64       
 2   High    700 non-null    float64       
 3   Low     700 non-null    float64       
 4   Open    700 non-null    float64       
 5   Volume  700 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 32.9 KB


In [101]:
print(df.head(), '\n',df.shape)

        Date      Close       High        Low       Open    Volume
0 2023-03-06  37.832508  38.542755  37.074912  37.832508  11198659
1 2023-03-08  39.442402  39.584449  37.785158  37.785158  16875862
2 2023-03-09  38.637447  39.679145  38.400698  39.679145   5386529
3 2023-03-10  38.874199  39.158297  37.785155  38.353352   5220866
4 2023-03-13  38.116600  38.968898  37.785153  38.826847   6647383 
 (700, 6)


In [102]:
df.head(6)

,Date,Close,High,Low,Open,Volume
0,2023-03-06,37.832508,38.542755,37.074912,37.832508,11198659
1,2023-03-08,39.442402,39.584449,37.785158,37.785158,16875862
2,2023-03-09,38.637447,39.679145,38.400698,39.679145,5386529
3,2023-03-10,38.874199,39.158297,37.785155,38.353352,5220866
4,2023-03-13,38.116600,38.968898,37.785153,38.826847,6647383
5,2023-03-14,38.069252,38.590099,37.785154,38.116601,5564064


In [103]:
target_col = "Close"
feature_cols = ["Open", "High", "Low", "Volume"]

In [104]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

In [105]:
WINDOW_SIZE = 2

In [106]:
def data_to_input_and_output(df, feature_cols, target_col):
    X = []
    y = []

    for i in range(len(df) - WINDOW_SIZE):
        # past WINDOW_SIZE rows, all features
        X.append(df[feature_cols].iloc[i:i+WINDOW_SIZE].values)

        # next step target
        y.append(df[target_col].iloc[i+WINDOW_SIZE])

    return np.array(X), np.array(y)


In [107]:
train_input, train_output = data_to_input_and_output(df, feature_cols=feature_cols, target_col=target_col)

In [108]:
print(train_input.shape, train_output.shape)

(698, 2, 4) (698,)


In [109]:
train_input[0]

array([[3.78325081e+01, 3.85427554e+01, 3.70749117e+01, 1.11986590e+07],
       [3.77851582e+01, 3.95844492e+01, 3.77851582e+01, 1.68758620e+07]])

In [110]:
train_output[0]

38.63744735717773

In [111]:
def train_neural_network(X, y, epochs=7, learning_rate=0.005):

    model = Sequential()
    model.add(InputLayer((WINDOW_SIZE, 4)))
    model.add(LSTM(64))
    model.add(Dense(8, 'relu'))
    model.add(Dense(1, 'linear'))

    check_point = ModelCheckpoint('model.h5', save_best_only=True, monitor='loss')
    model.compile(loss=MeanSquaredError(), optimizer=Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])

    model.fit(X, y, epochs=epochs, callbacks=[check_point])
    return model

In [114]:
model = train_neural_network(train_input, train_output, epochs=100, learning_rate=0.005)

Epoch 1/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step - loss: 5114.1831 - root_mean_squared_error: 71.5135

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5604.6562 - root_mean_squared_error: 74.8643
Epoch 2/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 4711.7598 - root_mean_squared_error: 68.6423

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5245.3408 - root_mean_squared_error: 72.4247 
Epoch 3/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 4414.3838 - root_mean_squared_error: 66.4408

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4708.7861 - root_mean_squared_error: 68.6206 
Epoch 4/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 4049.0562 - root_mean_squared_error: 63.6322

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3901.1228 - root_mean_squared_error: 62.4590 
Epoch 5/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 3434.1687 - root_mean_squared_error: 58.6018

...

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 361.3947 - root_mean_squared_error: 19.0104 
Epoch 15/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.5749 - root_mean_squared_error: 19.0151 
Epoch 16/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 308.6183 - root_mean_squared_error: 17.5675

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 361.3164 - root_mean_squared_error: 19.0083 
Epoch 17/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.7950 - root_mean_squared_error: 19.0209 
Epoch 18/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.4627 - root_mean_squared_error: 19.0122 
Epoch 19/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.4335 - root_mean_squared_error: 19.0114 
Epoch 20/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 476.8698 - root_mean_squared_error: 21.8373

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 361.2799 - root_mean_squared_error: 19.0074 
Epoch 21/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 361.4669 - root_mean_squared_error: 19.0123
Epoch 22/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.8261 - root_mean_squared_error: 19.0217 
Epoch 23/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.3012 - root_mean_squared_error: 19.0079 
Epoch 24/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.3327 - root_mean_squared_error: 19.0088 
Epoch 25/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 361.3529 - root_mean_squared_error: 19.0093 
Epoch 26/100
 1/22 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 351.9971 - root_mean_squared_error: 18.7616

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 361.2741 - root_mean_squared_error: 19.0072 
Epoch 27/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.4949 - root_mean_squared_error: 19.0130 
Epoch 28/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.3597 - root_mean_squared_error: 19.0095 
Epoch 29/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.4887 - root_mean_squared_error: 19.0129 
Epoch 30/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.4813 - root_mean_squared_error: 19.0127 
Epoch 31/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.8412 - root_mean_squared_error: 19.0221 
Epoch 32/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.8004 - root_mean_squared_error: 19.0211 
Epoch 33/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 362.1620 - root_mean_squared_error: 19.0306 
Epoch 34/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.9678 - root_mean_squared_error: 19.0255 
Epoch 35/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 361.4934

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 361.2121 - root_mean_squared_error: 19.0056 
Epoch 94/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.6150 - root_mean_squared_error: 19.0162 
Epoch 95/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.6309 - root_mean_squared_error: 19.0166 
Epoch 96/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 362.1812 - root_mean_squared_error: 19.0311 
Epoch 97/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 362.2881 - root_mean_squared_error: 19.0339 
Epoch 98/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.4363 - root_mean_squared_error: 19.0115 
Epoch 99/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.8733 - root_mean_squared_error: 19.0230 
Epoch 100/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 361.6592 - root_mean_squared_error: 19.0173 
